# 02 Silver Layer: Cleaning, Standardization & Enrichment

The **Silver Layer** serves as the single source of truth for downstream consumption.
In this notebook, we:
1. Ingest standardized raw records from `workspace.taxi_bronze.yellow_trips` and reference `taxi_zones`.
2. Audit data anomalies, missing values, and negative fares/distances.
3. Apply data quality filters (duration bounds, positive monetary fields, non-null timestamps).
4. Impute missing fields with defaults (e.g. `passenger_count`, surcharges).
5. Engineer analytical columns (`trip_duration_minutes`, `speed_mph`, `tip_pct`, `trip_date`, `pickup_hour`, etc.).
6. Enrich with Taxi Zone lookup dimensions (`pu_borough`, `pu_zone`, `do_borough`, `do_zone`).
7. Deduplicate records and persist to Delta table `workspace.taxi_silver.yellow_trips`.


In [0]:
import sys
# Add src to Python path for Databricks workspace execution or local session
sys.path.append("/Workspace/Users/yahyaghallali88@gmail.com/nyc_taxi-DB-DE/src")
sys.path.append("./src")

from pyspark.sql import functions as F
from taxi_lakehouse.silver import clean_yellow_trips, enrich_with_zones, build_silver_trips


## 1. Read Bronze Tables


In [0]:
bronze_base = "workspace.taxi_bronze"
yellow_path = f"{bronze_base}.yellow_trips"
zones_path = f"{bronze_base}.taxi_zones"

df_yellow = spark.read.table(yellow_path)
df_zones = spark.read.table(zones_path)

print(f"Bronze Yellow Trips total count: {df_yellow.count():,}")
print(f"Bronze Taxi Zones total count:   {df_zones.count():,}")
display(df_yellow.limit(5))


## 2. Bronze Data Quality Audit
Identify nulls, missing attributes, negative amounts, and date anomalies in raw bronze records.


In [0]:
print("--- Column Null Profile ---")
for c in df_yellow.columns:
    null_cnt = df_yellow.where(F.col(c).isNull()).count()
    if null_cnt > 0:
        print(f"{c:25s} : {null_cnt:,} nulls ({null_cnt / df_yellow.count():.2%})")

# Check for records with negative or zero fares/distances
non_pos_fare = df_yellow.where(F.col("fare_amount") <= 0).count()
non_pos_dist = df_yellow.where(F.col("trip_distance") <= 0).count()
print(f"\nNon-positive fare amount count:     {non_pos_fare:,}")
print(f"Non-positive trip distance count:    {non_pos_dist:,}")


In [0]:
# Inspect passenger_count and vendor distribution of null records
nulls = df_yellow.where(F.col("passenger_count").isNull())
print(f"Null passenger count rows: {nulls.count():,}")
display(nulls.groupBy("vendor_id", "payment_type").count().orderBy(F.desc("count")))


## 3. Apply Silver Transformation Pipeline
We invoke `build_silver_trips()` from `taxi_lakehouse.silver` to:
- Impute null passenger counts with 1
- Coalesce null surcharges to 0.0
- Filter out trips with invalid timestamps (constrain to Jan 2025 to drop year 2008/2088 glitches)
- Filter out non-positive fares, non-positive distances, and extreme durations (> 24 hours)
- Derive analytics features (`trip_duration_minutes`, `speed_mph`, `tip_pct`, `payment_type_desc`)
- Left-join pickup and dropoff zone metadata (`pu_borough`, `pu_zone`, `do_borough`, `do_zone`)
- Deduplicate identical trip records


In [0]:
df_silver = build_silver_trips(
    yellow_df=df_yellow,
    zones_df=df_zones,
    start_date="2025-01-01 00:00:00",
    end_date="2025-01-31 23:59:59"
)

display(df_silver.limit(10))


## 4. Validate Data Quality & Cleaning Impact


In [0]:
bronze_count = df_yellow.count()
silver_count = df_silver.count()
dropped_count = bronze_count - silver_count

print(f"Bronze raw count:     {bronze_count:,}")
print(f"Silver cleaned count: {silver_count:,}")
print(f"Dropped records:      {dropped_count:,} ({dropped_count / bronze_count:.2%})")

# Verify that essential columns have 0 nulls
print("\n--- Silver Quality Checks ---")
check_cols = ["passenger_count", "trip_duration_minutes", "pu_borough", "do_borough", "fare_amount", "speed_mph"]
for c in check_cols:
    c_nulls = df_silver.where(F.col(c).isNull()).count()
    print(f"{c:25s} : {c_nulls} nulls")


## 5. Persist to Silver Delta Table


In [0]:
silver_table = "workspace.taxi_silver.yellow_trips"

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Successfully written {silver_count:,} rows to {silver_table}")


## 6. Query Cleaned Silver Data


In [0]:
display(spark.sql(f"""
    SELECT 
        trip_date,
        pu_borough,
        COUNT(*) AS trip_count,
        ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(tip_pct), 2) AS avg_tip_pct
    FROM {silver_table}
    GROUP BY trip_date, pu_borough
    ORDER BY trip_date DESC, trip_count DESC
    LIMIT 20
"""))
